This notebook is the second iteration of "extract_training_data.ipynb". It differs from the original in that it generates multiple training examples per HealthBench question; one for every model that scored better than the target model. e.g. if we ran models A-E, and A is the target model, if models D and E both scored better than A on an input, we generate two examples, one where D's answer is the preferred answer, and one where E's answer is the preferred answer.

Original description:
For a target model, and for every HealthBench input in the training set, we want to grab the HealthBench input, and two completions: the best completion (regardless of which model produced it), and either the target model's completion or  the next worst completion if the target model had the best completion.

Target format:

```json
{"prompt": [{"role": "user", "content": "What color is the sky?"}],
 "chosen": [{"role": "assistant", "content": "It is blue."}],
 "rejected": [{"role": "assistant", "content": "It is green."}]}
```

In [ ]:
import json
from pathlib import Path
from typing import Callable

from simple_evals.improvement.models.results import AllResults
from simple_evals.improvement.models.benchmark_inputs import EvalInput
import polars as pl

In [ ]:
result_parent_dir = Path("../../../../results")
results_dirs = [
    Path("5df4ba309cb03369f6663786ae6a9904385524a9/maverick"),
    Path("5df4ba309cb03369f6663786ae6a9904385524a9/scout"),
    Path("328b8ce1d49f3834a8d91db014ca8de3e95e77f8/llama-3.3"),
    Path("9252fc34f9bc391262e8b71138ee3b86e7d0ad7a/o3"),
    Path("9eb82b46b5ef7faf6ec102a989421543e84d4228/llama-3.1-8b"),
    Path("cbd99b81af7e1cb59d122dec8d0cb78717b8d10d/llama-4-maverick-rag2"),
    Path("cbd99b81af7e1cb59d122dec8d0cb78717b8d10d/llama-4-scout-rag2"),
    Path("66a515a50edfaa2c8f21674d4141a124b50ef286/llama-4-maverick-enhanced-prompt"),
    Path("66a515a50edfaa2c8f21674d4141a124b50ef286/llama-4-scout-enhanced-prompt"),
    Path("a67ca8f9edea993fbb2551094c41777651cea1ec/enhanced_prompt_3"),
    Path("c6363d5c993ebf26e223714ba2210cb59372214d/context_awareness"),
    Path("7e027c10d0470439c600d35e8fa05e73ce618ae6/gpt-5"),
]
result_files = []
for path in results_dirs:
    full_path = result_parent_dir / path
    files = list(full_path.glob("*_allresults.json"))
    result_files.extend(files)

In [ ]:
def get_model_name_and_execution_time(results: Path) -> tuple[str, str]:
    _, model, day, time, _ = results.stem.split("_")
    return model, f"{day}_{time}"

In [ ]:
# columns
# model, run_id, prompt_id, completion, score
def get_results_df() -> pl.DataFrame:
    rows = []
    for result_file in result_files:
        file_results = AllResults.from_file(result_file)
        model_name, run_id = get_model_name_and_execution_time(result_file)
        for example_level_metadata in file_results.metadata.example_level_metadata:
            rows.append(
                {
                    "model": model_name,
                    "run_id": run_id,
                    "prompt_id": example_level_metadata.prompt_id,
                    "completion": example_level_metadata.completion[0].content,
                    "score": example_level_metadata.score,
                }
            )
    return pl.DataFrame(rows)


results = get_results_df()

In [ ]:
results

In [ ]:
# now filter the results to only prompt_id's in the training set
train_test_split = pl.read_csv("train_test.csv")
results_in_training_set = results.filter(
    pl.col("prompt_id").is_in(
        train_test_split.filter(pl.col("train_test") == "train")
        .select("prompt_id")
        .to_series()
        .to_list()
    )
)
results_in_training_set

In [ ]:
results_in_training_set.select(pl.col("model").unique())

In [ ]:
def get_training_rows(df: pl.DataFrame, target_model: str) -> list[dict]:
    rows = []
    for prompt_id_tup, group_df in df.group_by("prompt_id"):
        prompt_id = prompt_id_tup[0]
        target_model_row: dict | None = None
        better_performing_rows: list[dict] = []
        worse_performing_rows: list[dict] = []
        # First, find the highest_scoring row from the target model
        for row in group_df.iter_rows(named=True):
            if row["model"] == target_model:
                if target_model_row is None:
                    target_model_row = row
                elif row["score"] > target_model_row["score"]:
                    target_model_row = row
        if target_model_row is None:
            raise RuntimeError()

        # Now identify higher and lower-scoring rows, keeping only the highest
        # scoring response for each model (models tend to give very similar
        # responses across runs)
        def add_if_highest(list: list[dict], row: dict):
            present = False
            for i, item in enumerate(list):
                if item["model"] == row["model"]:
                    present = True
                    if row["score"] > item["score"]:
                        list[i] = row
            if not present:
                list.append(row)

        for row in group_df.iter_rows(named=True):
            if row["model"] == target_model:
                continue
            if row["score"] > target_model_row["score"]:
                add_if_highest(better_performing_rows, row)
            else:
                add_if_highest(worse_performing_rows, row)
        # Now sort the rows by score
        better_performing_rows.sort(key=lambda r: r["score"], reverse=True)
        worse_performing_rows.sort(key=lambda r: r["score"], reverse=True)
        if len(better_performing_rows) > 0:
            # There are models that scored better than the target model, create
            # one output row per better-performing row
            for better_performing_row in better_performing_rows:
                rows.append(
                    {
                        "prompt_id": prompt_id,
                        "preferred_model": better_performing_row["model"],
                        "preferred_run_id": better_performing_row["run_id"],
                        "preferred_completion": better_performing_row["completion"],
                        "preferred_score": better_performing_row["score"],
                        "rejected_model": target_model_row["model"],
                        "rejected_run_id": target_model_row["run_id"],
                        "rejected_completion": target_model_row["completion"],
                        "rejected_score": target_model_row["score"],
                    }
                )
        else:
            # The target model had the best score, only create one row
            rows.append(
                {
                    "prompt_id": prompt_id,
                    "preferred_model": target_model_row["model"],
                    "preferred_run_id": target_model_row["run_id"],
                    "preferred_completion": target_model_row["completion"],
                    "preferred_score": target_model_row["score"],
                    "rejected_model": worse_performing_rows[0]["model"],
                    "rejected_run_id": worse_performing_rows[0]["run_id"],
                    "rejected_completion": worse_performing_rows[0]["completion"],
                    "rejected_score": worse_performing_rows[0]["score"],
                }
            )
    return rows

In [ ]:
# For these test examples, the target model is "b".
test_df = pl.DataFrame(
    [
        # For the "first" prompt_id, we should get four examples: one each for
        # "a" (1), "x", "y", and "c" (2) where their response is preferred. In
        # all three cases the rejected response should be "b" from run_id 2.
        ("a", "1", "first", "blah", 1),
        ("x", "1", "first", "blah", 0.9),
        ("y", "1", "first", "blah", 0.8),
        ("b", "1", "first", "blah", 0.6),
        ("c", "1", "first", "blah", 0.5),
        ("a", "2", "first", "blah", 0.9),
        ("b", "2", "first", "blah", 0.7),
        ("c", "2", "first", "blah", 0.8),
        # For the "second" prompt_id the best response should be "b"'s and the
        # rejected response should be "c"'s.
        ("a", "1", "second", "blah", 0.3),
        ("b", "1", "second", "blah", 0.7),
        ("c", "1", "second", "blah", 0.6),
        # For the "third" same outcome as the first, but we should get two
        # examples, one for "a", and one for "c".
        ("a", "1", "third", "blah", 1),
        ("b", "1", "third", "blah", 0.6),
        ("c", "1", "third", "blah", 0.7),
    ],
    schema=[
        ("model", pl.String),
        ("run_id", pl.String),
        ("prompt_id", pl.String),
        ("completion", pl.String),
        ("score", pl.Float32),
    ],
)

expected = [
    ("first", "a", "1", "blah", "b", "2", "blah"),
    ("first", "x", "1", "blah", "b", "2", "blah"),
    ("first", "y", "1", "blah", "b", "2", "blah"),
    ("first", "c", "2", "blah", "b", "2", "blah"),
    ("second", "b", "1", "blah", "c", "1", "blah"),
    ("third", "a", "1", "blah", "b", "1", "blah"),
    ("third", "c", "1", "blah", "b", "1", "blah"),
]

test_selections = get_training_rows(test_df, "b")
test_selections
# test_selections = test_df.group_by("prompt_id").map_groups(
#     process_question_group_factory(target_model="b")
# )
# test_selections

In [ ]:
# Check that all of the expected test rows are present
assert len(test_selections) == len(expected), (
    f"{len(test_selections)} == {len(expected)}"
)
for expected_row in expected:
    (
        expected_prompt_id,
        expected_preferred_model,
        expected_preferred_run_id,
        expected_preferred_completion,
        expected_rejected_model,
        expected_rejected_run_id,
        expected_rejected_completion,
    ) = expected_row
    match = False
    for selection in test_selections:
        if (
            (selection["preferred_model"] == expected_preferred_model)
            and (selection["preferred_run_id"] == expected_preferred_run_id)
            and (selection["rejected_model"] == expected_rejected_model)
            and (selection["rejected_run_id"] == expected_rejected_run_id)
        ):
            match = True
            break
    if not match:
        raise KeyError(expected_row)

In [ ]:
target_model = "llama-4-maverick"
selections = get_training_rows(results_in_training_set, target_model)
selections_df = pl.DataFrame(selections)
selections_df

In [ ]:
# only keep examples where the preferred response is from the target model, or
# where the preferred response is at least 50% better than the target model's
# response
selections_df_filtered = selections_df.with_columns(
    (
        pl.col("preferred_score")
        / pl.when(pl.col("rejected_score") == 0)
        .then(pl.lit(0.0000000001))
        .otherwise(pl.col("rejected_score"))
    ).alias("score_diff")
).filter((pl.col("preferred_model") == target_model) | (pl.col("score_diff") >= 1.5))
selections_df_filtered

In [ ]:
selections_df_filtered.filter(pl.col("preferred_model") == target_model).shape

In [ ]:
selections_df_filtered.shape

In [ ]:
selections_df_filtered.select(pl.col("preferred_model").value_counts()).unnest(
    "preferred_model"
).sort("count", descending=True)

In [ ]:
# Now get the inputs and construct the final dataset
eval_inputs = EvalInput.from_inputs(
    Path("../../../../results/inputs/2025-05-07-06-14-12_oss_eval.jsonl")
)
inputs_by_prompt_id = {eval_input.prompt_id: eval_input for eval_input in eval_inputs}

In [ ]:
objects = []
for row in selections_df_filtered.iter_rows(named=True):
    input = inputs_by_prompt_id[row["prompt_id"]]
    # convert the input prompt turns to dicts
    turns = []
    for turn in input.prompt:
        turns.append(
            {
                "content": turn.content,
                "role": turn.role.value,
            }
        )
    objects.append(
        {
            "metadata": {
                "prompt_id": input.prompt_id,
                "preferred_model": row["preferred_model"],
                "preferred_run_id": row["preferred_run_id"],
                "rejected_model": row["rejected_model"],
                "rejected_run_id": row["rejected_run_id"],
            },
            "prompt": turns,
            "chosen": row["preferred_completion"],
            "rejected": row["rejected_completion"],
        }
    )
for object in objects:
    if len(object["prompt"]) == 1:
        continue
    o = object
    break
print(json.dumps(o, indent=2))

In [ ]:
with Path("training_data_enhanced.jsonl").open("a") as file:
    for object in objects:
        file.write(json.dumps(object))
        file.write("\n")